# Test RAG — Recherche dans la knowledge base

Ce notebook permet de tester le RAG manuellement :
- Charger un fichier ou saisir une requête
- Générer l'embedding de la requête
- Rechercher les chunks les plus proches dans Qdrant
- Afficher les résultats avec leur score de similarité

In [ ]:
# ── Dépendances ──────────────────────────────────────────────────────────────
# pip install openai qdrant-client python-dotenv

import os
from pathlib import Path

from dotenv import load_dotenv

# Charge le .env à la racine du projet
load_dotenv(Path("../").resolve() / ".env")

OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
QDRANT_URL       = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION_NAME  = os.getenv("COLLECTION_NAME", "offria_kb")
EMBEDDING_MODEL  = "text-embedding-3-small"
EMBEDDING_DIM    = 1536

print(f"OPENAI_API_KEY : {'OK' if OPENAI_API_KEY else 'MANQUANTE'}")
print(f"Qdrant URL     : {QDRANT_URL}")
print(f"Collection     : {COLLECTION_NAME}")

In [ ]:
# ── Connexion Qdrant + OpenAI ────────────────────────────────────────────────
import openai
from qdrant_client import QdrantClient

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
qdrant        = QdrantClient(url=QDRANT_URL)

# Vérifie que la collection existe
collections = [c.name for c in qdrant.get_collections().collections]
print(f"Collections disponibles : {collections}")
if COLLECTION_NAME in collections:
    info = qdrant.get_collection(COLLECTION_NAME)
    print(f"Vecteurs dans '{COLLECTION_NAME}' : {info.points_count}")
else:
    print(f"⚠️  Collection '{COLLECTION_NAME}' introuvable — lance l'indexation d'abord")

In [ ]:
# ── Option A : requête texte libre ──────────────────────────────────────────
QUERY = "approche méthodologique suivi évaluation"
TOP_K = 5  # nombre de résultats

print(f"Requête : {QUERY}")

In [ ]:
# ── Option B : charger un fichier comme requête ──────────────────────────────
# Décommente et adapte le chemin pour utiliser un fichier comme contexte de recherche

# FILE_PATH = Path("../knowledge_base/company/presentation_abi_consulting.md")
# QUERY = FILE_PATH.read_text(encoding="utf-8")[:2000]  # 2000 premiers caractères
# print(f"Fichier chargé : {FILE_PATH.name} ({len(QUERY)} chars)")

In [ ]:
# ── Embedding de la requête ──────────────────────────────────────────────────
response = openai_client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=QUERY,
)
query_vector = response.data[0].embedding
print(f"Embedding généré : {len(query_vector)} dimensions")

In [ ]:
# ── Recherche dans Qdrant ────────────────────────────────────────────────────
results = qdrant.search(
    collection_name=COLLECTION_NAME,
    query_vector=query_vector,
    limit=TOP_K,
    with_payload=True,
)

print(f"\n{'='*60}")
print(f"Top {TOP_K} résultats pour : '{QUERY}'")
print(f"{'='*60}\n")

for i, hit in enumerate(results, 1):
    print(f"[{i}] Score : {hit.score:.4f}")
    print(f"    Fichier  : {hit.payload.get('rel_path')}")
    print(f"    Type     : {hit.payload.get('doc_type')}")
    print(f"    Chunk #{hit.payload.get('chunk_idx')}")
    print(f"    Contenu  :")
    content = hit.payload.get('content', '')
    print(f"      {content[:300]}{'...' if len(content) > 300 else ''}")
    print()

In [ ]:
# ── Filtrer par type de document ────────────────────────────────────────────
# Utile pour chercher uniquement dans les références, templates, etc.

from qdrant_client.models import Filter, FieldCondition, MatchValue

DOC_TYPE_FILTER = "references"  # company | references | templates | resources | certifications

results_filtered = qdrant.search(
    collection_name=COLLECTION_NAME,
    query_vector=query_vector,
    limit=TOP_K,
    with_payload=True,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="doc_type",
                match=MatchValue(value=DOC_TYPE_FILTER),
            )
        ]
    ),
)

print(f"\n{'='*60}")
print(f"Résultats filtrés sur doc_type='{DOC_TYPE_FILTER}'")
print(f"{'='*60}\n")

for i, hit in enumerate(results_filtered, 1):
    print(f"[{i}] Score : {hit.score:.4f} | {hit.payload.get('rel_path')}")
    content = hit.payload.get('content', '')
    print(f"    {content[:200]}{'...' if len(content) > 200 else ''}")
    print()

In [ ]:
# ── Stats de la collection ───────────────────────────────────────────────────
from collections import Counter

# Récupère tous les points (scroll)
all_points, _ = qdrant.scroll(
    collection_name=COLLECTION_NAME,
    limit=10000,
    with_payload=True,
    with_vectors=False,
)

by_file = Counter(p.payload.get('rel_path') for p in all_points)
by_type = Counter(p.payload.get('doc_type') for p in all_points)

print(f"Total chunks : {len(all_points)}\n")
print("Par type :")
for doc_type, count in sorted(by_type.items()):
    print(f"  {doc_type:<30} : {count} chunks")

print("\nPar fichier :")
for rel_path, count in sorted(by_file.items()):
    print(f"  {rel_path:<50} : {count} chunks")